In [1]:
#@title Environent Setup
!pip install --quiet optuna
!pip install --quiet torch

# Connect to drive
from google.colab import drive, userdata
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/ML4Finance/scripts')
import torch
import optuna
import pandas as pd
import RNN
import Embedders
import os
import json
import Trainer
import dataProcessor as dP  # Your dataset and dataloader module
from torch.utils.data import DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
#@title Training setup

#—————————————————————————————————————————————————————#
device         = 'cuda'
dataset_path   = '/content/drive/MyDrive/ML4Finance/onion_dataset.parquet'
embedding_path = '/content/drive/MyDrive/ML4Finance/trained_model/mds_embeddings.csv'
#—————————————————————————————————————————————————————#
# Load data
batch_size     = 1024 # @param ["32","64","128","256","512","1024"] {"type":"raw"}
seq_len        = 168 # @param ["168", "336"] {"type":"raw","placeholder":"hours"}
prediction_len = 24

#—————————————————————————————————————————————————————#
# Obtain basically everything
processed, scalers, zones, hours, dows, months, feature_names = Trainer.preprocess_and_scale(dataset_path)

# Set input size as prices + number of features
input_size = len(feature_names)

# Build dataset
dataset = Trainer.CustomTensorDataset(
    scaled_data_dict = processed,
    hours    = hours,
    dows     = dows,
    months   = months,
    seq_len  = seq_len,
    pred_len = prediction_len
)

# Split dataset
train_size = int(0.8 * len(dataset))
train_set = torch.utils.data.Subset(dataset, list(range(train_size)))
val_set   = torch.utils.data.Subset(dataset, list(range(train_size, len(dataset))))

# Create dataloaders
train_dataloader = DataLoader(train_set, batch_size = batch_size, shuffle=True,  pin_memory=True, num_workers=2)
val_dataloader   = DataLoader(val_set, batch_size   = batch_size, shuffle=False, pin_memory=True, num_workers=2)

#—————————————————————————————————————————————————————#
# Obtain embeddings
mds_df = pd.read_csv(embedding_path, index_col=0)
pretrained_bzn = torch.tensor(
    mds_df.loc[zones].values,   # reorder
    dtype  = torch.float32,
    device = device
)
# Dimensions
n_bzn, bzn_dim = pretrained_bzn.shape

#—————————————————————————————————————————————————————#
bzn_emb = Embedders.BZNEmbedder(
    n_bzn      = n_bzn,
    bzn_dim    = bzn_dim,
    pretrained = pretrained_bzn,
    freeze     = False ,# @param {"type":"boolean","placeholder":"Freeze BZN Embedding"}
    device     = device
).weight

#—————————————————————————————————————————————————————#
# Output dimension -> number of hours predicted
output_size = prediction_len


KeyError: "['IT-north'] not in index"

In [ ]:
#@title Single Training
hidden_size   = 387  # @param {"type":"integer"}
epochs        = 18   # @param {"type":"integer"}
learning_rate = 1e-3
dropout_rate  = 0.15
num_layers    = 3
device        = 'cuda'

#———— Get embeddings
ts_emb  = Embedders.TimestampEmbedder(d_hour = 8, d_day = 4, d_month = 4, freeze = True, device = device)
bzn_emb = Embedders.BZNEmbedder(n_bzn = n_bzn, bzn_dim = pretrained_bzn.shape[-1], pretrained = pretrained_bzn, freeze = False, device = device)

model   = RNN.DeepLSTM(
    input_size    = input_size,
    hidden_size   = hidden_size,
    output_size   = 24,
    time_embedder = ts_emb,
    bzn_embedder  = bzn_emb,
    num_layers    = num_layers,
    dropout       = dropout_rate,
    device        = device
).to(device)

#———— Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)
#———— Loss function
loss_function = lambda x, y: Trainer.Weighted_MSE(x, y)


#———— Training
train_history, val_history = Trainer.training_loop(
    model            = model,
    optimizer        = optimizer,
    loss_function    = loss_function,
    train_dataloader = train_dataloader,
    val_loader       = val_dataloader,
    epochs           = epochs,
    device           = device
)

# Create output directory
output_dir = "drive/MyDrive/ML4Finance/trained_model"
os.makedirs(output_dir, exist_ok = True)

# Save model weights
torch.save(model.state_dict(), os.path.join(output_dir, "lstm_weights.pt"))

print("Training run complete. Model and hyperparameters saved.")

with torch.no_grad():                         # safety: no grad tracking
    bzn_vectors = model.get_bzn_vectors()     # shape = (n_bzn, bzn_dim)

torch.save(bzn_vectors.cpu(), '/content/drive/MyDrive/ML4Finance/trained_model/trained_embedding.pt')

In [ ]:
#@title Autoclicker
from IPython.display import Javascript

# Prevents idle timeout by simulating a click in the notebook interface
Javascript('''
function ClickConnect(){
    console.log("Clicking on reconnect button to prevent timeout");
    var buttons = [...document.querySelectorAll("colab-toolbar-button")];
    buttons.forEach(btn => {
        if (btn.innerText.includes("Reconnect") || btn.innerText.includes("Reconnect to runtime")) {
            btn.click();
        }
    });
}
setInterval(ClickConnect, 60000); // every 60 seconds
''')

<IPython.core.display.Javascript object>

In [ ]:
#@title Hyperparameter Optimization Study

epochs     = 10 # @param {"type":"integer"}
num_trials = 50 # @param {"type":"integer"}

study = Trainer.lstm_study(
    num_trials       = num_trials,
    output_filepath  = "best_hyperparams.csv",
    train_dataloader = train_dataloader,
    val_dataloader   = val_dataloader,
    n_bzn            = n_bzn,
    input_size       = input_size,
    pretrained_embed = bzn_emb,
    epochs           = epochs
)

print("Best hyperparameters:", study.best_params)
print("Best validation loss:", study.best_value)

[I 2025-05-29 10:15:45,886] A new study created in memory with name: no-name-a2a8f033-7b09-4c12-ae9d-818c8e1fa34e


  0%|          | 0/50 [00:00<?, ?it/s]

Epoch:  1  |  Train: 0.421460  |  Val: 0.410157  |  Time: 116.20s
Epoch:  2  |  Train: 0.205582  |  Val: 0.403567  |  Time: 119.48s
Epoch:  3  |  Train: 0.175556  |  Val: 0.389927  |  Time: 120.25s
Epoch:  4  |  Train: 0.148678  |  Val: 0.393517  |  Time: 121.35s
Epoch:  5  |  Train: 0.122884  |  Val: 0.403320  |  Time: 121.53s
Stopping early at epoch 5
[I 2025-05-29 10:25:44,761] Trial 0 finished with value: 0.4033195344025938 and parameters: {'hidden_size': 302, 'num_layers': 3}. Best is trial 0 with value: 0.4033195344025938.
Epoch:  1  |  Train: 0.422273  |  Val: 0.447840  |  Time: 278.04s
Epoch:  2  |  Train: 0.206438  |  Val: 0.387018  |  Time: 281.89s
Epoch:  3  |  Train: 0.179005  |  Val: 0.389760  |  Time: 281.64s
Epoch:  4  |  Train: 0.148459  |  Val: 0.385781  |  Time: 281.71s
Epoch:  5  |  Train: 0.127677  |  Val: 0.391499  |  Time: 282.52s
Epoch:  6  |  Train: 0.104555  |  Val: 0.396424  |  Time: 283.10s
Stopping early at epoch 6
[I 2025-05-29 10:53:53,903] Trial 1 finishe

In [ ]:
import matplotlib.pyplot as plt
def show_LSTM_study_results(study):

    # Set the size of the figures
    fig = plt.figure(figsize=(15, 10))

    # Plot optimization history
    fig = optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.savefig('drive/MyDrive/ML4Finance/trained_model/LSTM_opt_history.png')
    # Plot hyperparameter importance
    fig = optuna.visualization.matplotlib.plot_param_importances(study)
    plt.savefig('drive/MyDrive/ML4Finance/trained_model/LSTM_opt_paramRelations.png')
    # Plot hyperparameter relationships
    fig = optuna.visualization.matplotlib.plot_slice(study)
    plt.savefig('drive/MyDrive/ML4Finance/trained_model/LSTM_opt_slice.png')
    # Plot contour of hyperparameters
    fig = optuna.visualization.matplotlib.plot_contour(study)
    plt.savefig('drive/MyDrive/ML4Finance/trained_model/LSTM_opt_contourPlot.png')

show_LSTM_study_results(study)

In [ ]:
plot_predictions_per_bzn(
    model        = model,
    val_dataset  = val_set,
    scalers      = scalers,
    bzn_names    = bzn_names,
    seq_len      = seq_len,
    pred_len     = 24,
    max_steps    = 200,  # ~200 days
    device       = "cuda",
    output_dir   = "drive/MyDrive/ML4Finance/trained_model/forecasts"
)
